# AMPS II · Week 09 — HW8
## Shells: facets, supports and symmetry · homogenization: singularities, cell BCs and an LLM-generated script

**Due before Week 10 · 100 points.** Submit the executed notebook: *Runtime → Restart session and run all*, fill in the AI-usage log at the end.
Every number you report must come from a cell of this notebook. The first cell of Q7 runs an LLM-generated script that is **meant** to be wrong.
The setup cell installs `dolfinx_mpc` (≈ 1 min); the whole HW runs in a few minutes.

| Q | Topic | Points |
|---|---|:---:|
| 1 | Theory: drilling, the mean-strain theorem, why cell BCs bound the answer | 15 |
| 2 | A pressurized tube: the flat-facet error | 12 |
| 3 | Clamping an I-beam: what a marker selects | 15 |
| 4 | The Scordelis–Lo roof on a quarter with symmetry conditions | 15 |
| 5 | The checkerboard: a corner singularity and a graded mesh | 15 |
| 6 | A porous medium: cell BCs and window size | 13 |
| 7 | Debugging an LLM-generated homogenization script | 15 |

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and dolfinx_mpc and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

# dolfinx_mpc 0.11 (periodic constraints) into the same environment — ≈ 1 min, once per runtime (2 packages, nothing else changes)
import glob
if not glob.glob('/content/micromamba/envs/fenicsx/lib/python3*/site-packages/dolfinx_mpc'):
    subprocess.run('/content/micromamba/bin/micromamba install -y -q -n fenicsx -c conda-forge dolfinx_mpc=0.11', shell=True, check=True,
                   env=dict(os.environ, MAMBA_ROOT_PREFIX='/content/micromamba'))
print('dolfinx_mpc installed:', bool(glob.glob('/content/micromamba/envs/fenicsx/lib/python3*/site-packages/dolfinx_mpc')))

In [ ]:
%%fenicsx --info -np 2

The model modules from the lab (unchanged). Run the two cells once.

In [ ]:
%%writefile /tmp/fnx/shell.py
"""Linear Reissner-Mindlin shell on a triangulated surface in 3D, DOLFINx 0.11 (the source's formulation).
Unknowns: u (3 displacement components, P2) and theta (rotation vector, 3 Cartesian components, Crouzeix-Raviart).
Local frame (e1, e2, e3) per cell (DG0) from the cell Jacobian; P = [e1 | e2] (3 x 2); the tangential gradient is grad(.) P.
    membrane  eps   = sym(P^T grad(u) P)                    N = t C eps
    bending   kappa = sym(P^T grad(beta) P), beta = e3 x theta    M = t^3/12 C kappa      (C: plane stress)
    shear     gamma = grad(u . e3) P - P^T beta              Q = mu t gamma   (shear factor 1, as in the source)
    drilling  E t^3 / h^2 [ ((P^T grad u P)_12 - (P^T grad u P)_21)/2 + theta . e3 ]^2   (penalty: theta . e3 has no stiffness otherwise)
Meshes are built from point arrays with mesh.create_mesh (no Gmsh): plate, cylinder, Scordelis-Lo roof, straight I-beam."""
import numpy as np
import ufl, basix.ufl
from mpi4py import MPI
from dolfinx import fem, mesh, geometry
from dolfinx.fem.petsc import LinearProblem

comm = MPI.COMM_WORLD
def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

LU = {"ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"}

def surface_mesh(X, cells):
    """X: (n, 3) points, cells: (m, 3) triangles -> a mesh of 2D cells in 3D (rank 0 passes the arrays, the others empty arrays)"""
    el = ufl.Mesh(basix.ufl.element("Lagrange", "triangle", 1, shape=(3,)))
    if comm.rank != 0:
        X, cells = np.zeros((0, 3)), np.zeros((0, 3), dtype=np.int64)
    return mesh.create_mesh(comm, np.asarray(cells, dtype=np.int64), el, np.asarray(X, dtype=np.float64))

def grid(n1, n2, f):
    """structured triangulation of the parameter square [0, 1]^2 (alternating diagonals), mapped by f(s, t) -> (x, y, z)"""
    s, t = np.meshgrid(np.linspace(0, 1, n1 + 1), np.linspace(0, 1, n2 + 1), indexing="ij")
    X = np.array(f(s.ravel(), t.ravel())).T
    idx = lambda i, j: i * (n2 + 1) + j; C = []
    for i in range(n1):
        for j in range(n2):
            a, b, c, d = idx(i, j), idx(i + 1, j), idx(i + 1, j + 1), idx(i, j + 1)
            C += [[a, b, c], [a, c, d]] if (i + j) % 2 == 0 else [[a, b, d], [b, c, d]]
    return X, np.array(C)

def plate_mesh(N, R=np.eye(3), L=1.0):
    """the square [0, L]^2 in the plane z = 0, then rotated by the matrix R"""
    X, C = grid(N, N, lambda s, t: (L * s, L * t, 0 * s))
    return surface_mesh(X @ np.asarray(R).T, C)

def cylinder_mesh(nth, nz, R=1.0, H=2.0):
    """closed tube of radius R about the z axis, -H/2 < z < H/2 (the seam is merged: a periodic parameter direction)"""
    X, C = grid(nth, nz, lambda s, t: (R * np.cos(2 * np.pi * s), R * np.sin(2 * np.pi * s), H * (t - 0.5)))
    n2 = nz + 1
    C = np.where(C >= nth * n2, C - nth * n2, C)
    return surface_mesh(X[: nth * n2], C)

def scordelis_lo_mesh(N, L=50.0, R=25.0, phi=40.0, quarter=False):
    """the Scordelis-Lo roof: cylinder axis x, -L/2 < x < L/2, angle -phi..phi from the crown (crown at y = 0, z = R), N x N cells;
    quarter=True: 0 < x < L/2 and 0 < angle < phi (symmetry planes x = 0 and y = 0), N x N cells"""
    p = np.deg2rad(phi)
    if quarter:
        X, C = grid(N, N, lambda s, t: (L / 2 * s, R * np.sin(p * t), R * np.cos(p * t)))
    else:
        X, C = grid(N, N, lambda s, t: (L * (s - 0.5), R * np.sin(p * (2 * t - 1)), R * np.cos(p * (2 * t - 1))))
    return surface_mesh(X, C)

def ibeam_mesh(nx, nf, nw, L=10.0, b=0.2, hw=0.4):
    """straight I-beam along x: two flanges (width b, at z = 0 and z = hw) and a web (height hw, in y = 0). The web edges ARE the flange centre
    lines (shared vertices): a non-manifold junction where three cells meet at an edge. nf cells across each half flange, nw over the web."""
    pts, ids, C = [], {}, []
    def P(i, y, z):
        k = (i, round(y, 12), round(z, 12))
        if k not in ids:
            ids[k] = len(pts); pts.append((L * i / nx, y, z))
        return ids[k]
    def strip(ys, zs):
        for i in range(nx):
            for j in range(len(ys) - 1):
                a, b_, c, d = P(i, ys[j], zs[j]), P(i + 1, ys[j], zs[j]), P(i + 1, ys[j + 1], zs[j + 1]), P(i, ys[j + 1], zs[j + 1])
                C.extend([[a, b_, c], [a, c, d]] if (i + j) % 2 == 0 else [[a, b_, d], [b_, c, d]])
    yf = np.linspace(-b / 2, b / 2, 2 * nf + 1)
    strip(yf, np.zeros_like(yf)); strip(yf, np.full_like(yf, hw))
    zw = np.linspace(0, hw, nw + 1); strip(np.zeros_like(zw), zw)
    return surface_mesh(np.array(pts), np.array(C))

def normalize(v):
    return v / ufl.sqrt(ufl.dot(v, v))

def local_frame(msh):
    """e3 = unit normal of the (flat) cell, e1 = ey x e3 (ez if e3 is nearly along y), e2 = e3 x e1 — the source's choice"""
    t = ufl.Jacobian(msh)
    e3 = normalize(ufl.cross(ufl.as_vector([t[0, 0], t[1, 0], t[2, 0]]), ufl.as_vector([t[0, 1], t[1, 1], t[2, 1]])))
    e1 = ufl.cross(ufl.as_vector([0.0, 1.0, 0.0]), e3)
    e1 = ufl.conditional(ufl.lt(ufl.sqrt(ufl.dot(e1, e1)), 0.5), ufl.as_vector([0.0, 0.0, 1.0]), normalize(e1))
    return e1, normalize(ufl.cross(e3, e1)), e3

class Shell:
    count = 0
    def __init__(self, msh, t=1e-3, E=210e9, nu=0.3, drill=1.0, rot="CR"):
        """t thickness; drill = factor on the drilling stiffness E t^3 / h^2 (0: none); rot = "CR" (the source) or "P2" for the rotations"""
        self.msh, self.t, self.E, self.nu = msh, t, E, nu
        cell = msh.basix_cell()
        eth = basix.ufl.element("CR", cell, 1, shape=(3,)) if rot == "CR" else basix.ufl.element("Lagrange", cell, 2, shape=(3,))
        V = self.V = fem.functionspace(msh, basix.ufl.mixed_element([basix.ufl.element("Lagrange", cell, 2, shape=(3,)), eth]))
        VT = fem.functionspace(msh, ("DG", 0, (3,)))
        self.frame = []
        for ei in local_frame(msh):                                # interpolate the frame into DG0: constant per cell
            f_ = fem.Function(VT); f_.interpolate(fem.Expression(ei, VT.element.interpolation_points)); self.frame.append(f_)
        e1, e2, e3 = self.frame
        P = ufl.as_matrix([[e1[i], e2[i]] for i in range(3)])
        tg = lambda w: ufl.dot(ufl.grad(w), P)
        lam, mu = E * nu / (1 + nu) / (1 - 2 * nu), E / 2 / (1 + nu); lps = 2 * lam * mu / (lam + 2 * mu)
        Cps = lambda e: lps * ufl.tr(e) * ufl.Identity(2) + 2 * mu * e
        def strains(u, th):
            gu = ufl.dot(P.T, tg(u)); beta = ufl.cross(e3, th)
            return ufl.sym(gu), ufl.sym(ufl.dot(P.T, tg(beta))), tg(ufl.dot(u, e3)) - ufl.dot(P.T, beta), (gu[0, 1] - gu[1, 0]) / 2 + ufl.dot(th, e3)
        self.strains = strains
        (eU, kU, gU, dU), (eW, kW, gW, dW) = strains(*ufl.TrialFunctions(V)), strains(*ufl.TestFunctions(V))
        dx = self.dx = ufl.Measure("dx", domain=msh); h = ufl.CellDiameter(msh)
        self.a = (t * ufl.inner(Cps(eU), eW) + t**3 / 12 * ufl.inner(Cps(kU), kW) + mu * t * ufl.dot(gU, gW)
                  + drill * E * t**3 / h**2 * dU * dW) * dx
        self.w = fem.Function(V); self.bcs = []; self.X = ufl.SpatialCoordinate(msh)

    def fix(self, where, sub, comps=None):
        """fix u (sub 0) or theta (sub 1), all components (comps None) or a list of components, on
        where = a marker lambda x: bool (boundary facets), an array of facet indices, or ("points", marker) for vertex dofs of u"""
        V = self.V.sub(sub); fdim = self.msh.topology.dim - 1
        if isinstance(where, tuple):
            for c in (comps if comps is not None else range(3)):
                Vc, _ = V.sub(c).collapse()
                self.bcs.append(fem.dirichletbc(fem.Function(Vc), fem.locate_dofs_geometrical((V.sub(c), Vc), where[1]), V.sub(c)))
            return
        fac = where if isinstance(where, np.ndarray) else mesh.locate_entities_boundary(self.msh, fdim, where)
        for c in (comps if comps is not None else [None]):
            S = V if c is None else V.sub(c); Vc, _ = S.collapse()
            self.bcs.append(fem.dirichletbc(fem.Function(Vc), fem.locate_dofs_topological((S, Vc), fdim, fac), S))
        return comm.allreduce(len(fac), op=MPI.SUM)

    def solve(self, f):
        """f: UFL vector, force per unit area. Returns the KSP reason (MUMPS on a singular matrix may still return 'converged' with NaN)."""
        Shell.count += 1
        pb = LinearProblem(self.a, ufl.dot(f, ufl.TestFunctions(self.V)[0]) * self.dx, u=self.w, bcs=self.bcs, petsc_options=LU,
                           petsc_options_prefix=f"shell{Shell.count}_")
        pb.solve()
        return pb.solver.getConvergedReason()

    def u(self):
        return self.w.sub(0).collapse()

    def at(self, p):
        """displacement vector at the point p (evaluated in the first cell owned by this rank that contains it)"""
        u = self.u(); msh = self.msh; tdim = msh.topology.dim; pp = np.array([p], dtype=np.float64)
        c = geometry.compute_colliding_cells(msh, geometry.compute_collisions_points(geometry.bb_tree(msh, tdim), pp), pp).links(0)
        own = [k for k in c if k < msh.topology.index_map(tdim).size_local]
        val = np.asarray(u.eval(pp, [own[0]])).reshape(-1) if own else np.zeros(3); n = 1 if own else 0
        return comm.allreduce(val * n, op=MPI.SUM) / max(comm.allreduce(n, op=MPI.SUM), 1)

    def umax(self):
        """max |u_i| over the dofs, per component"""
        a = self.u().x.array.reshape(-1, 3)
        return np.array([comm.allreduce(float(np.abs(a[:, i]).max()) if len(a) else 0.0, op=MPI.MAX) for i in range(3)])

    def energy(self):
        """strain energy 1/2 a(w, w)"""
        return 0.5 * comm.allreduce(fem.assemble_scalar(fem.form(ufl.action(ufl.action(self.a, self.w), self.w))), op=MPI.SUM)

    def ndofs(self):
        return self.V.dofmap.index_map.size_global * self.V.dofmap.index_map_bs

def navier_ss(D, F=np.inf, q=1.0, a=1.0, nterms=400):
    """centre deflection of the simply supported square, Kirchhoff (F = inf) or the hard simply supported Reissner-Mindlin plate:
    w_K + M/F, M = (Mxx + Myy)/(1 + nu) solves -lap M = q, M = 0 on the edges (Week 8)"""
    wk = m_ = 0.0
    for m in range(1, nterms, 2):
        for n in range(1, nterms, 2):
            s = np.sin(m * np.pi / 2) * np.sin(n * np.pi / 2); k2 = (m**2 + n**2) * np.pi**2 / a**2; c = 16 * q / (np.pi**2 * m * n)
            wk += c / (D * k2**2) * s; m_ += c / k2 * s
    return wk + (m_ / F if np.isfinite(F) else 0.0)

def beam_clamped(x, q, L, EI, GAs=np.inf):
    """clamped-clamped Timoshenko beam under a uniform load q: w(x) = q x^2 (L - x)^2 / (24 EI) + q x (L - x) / (2 G A_s)"""
    x = np.asarray(x); return q * x**2 * (L - x)**2 / (24 * EI) + (q * x * (L - x) / (2 * GAs) if np.isfinite(GAs) else 0.0)

In [ ]:
%%writefile /tmp/fnx/homog.py
"""Periodic homogenization of 2D unit cells with dolfinx_mpc, DOLFINx 0.11.
Elasticity (plane strain): v periodic,   C (E + eps(v)) : eps(w) = 0   ->   C_hom[:, k] = < sigma(E_k + eps(v_k)) >   (Voigt, E = (E11, E22, 2 E12))
Conduction:                phi periodic, k (G + grad phi) . grad psi = 0 ->  K_hom[:, k] = < k (G_k + grad phi_k) >
< . > = (1 / |cell|) * integral over the MESH — |cell| = Lx Ly, also when the mesh has holes (voids).
Cell boundary conditions: "periodic" (dolfinx_mpc), "kinematic" (fluctuation 0 on the boundary), "static" (uniform traction / flux).
Two dolfinx_mpc rules (both silent if broken):
  * the corner (Lx, Ly) must be constrained once (to (0, 0)), not by both side constraints;
  * pin the rigid translation at an INTERIOR vertex: a Dirichlet condition on a MASTER dof of a periodic constraint is ignored."""
import numpy as np
import ufl
from mpi4py import MPI
from dolfinx import fem, mesh
import dolfinx_mpc

comm = MPI.COMM_WORLD
def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

LU = {"ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"}
TOL = 1e-8

# ---------------------------------------------------------------- cells: (mesh, phase indicator ph (DG0, 1 = phase 2), (Lx, Ly))
def indicator(msh, inside):
    Q = fem.functionspace(msh, ("DG", 0)); ph = fem.Function(Q)
    n = msh.topology.index_map(msh.topology.dim).size_local
    mid = mesh.compute_midpoints(msh, msh.topology.dim, np.arange(n, dtype=np.int32))
    ph.x.array[:n] = inside(mid.T).astype(float); ph.x.scatter_forward()
    return ph

def laminate_cell(f2, ny=20, nx=2):
    """unit square, layers normal to y: phase 2 for y < f2 (an interface on a mesh line if f2 ny is an integer)"""
    msh = mesh.create_unit_square(comm, nx, ny, mesh.CellType.quadrilateral)
    return msh, indicator(msh, lambda x: x[1] < f2), (1.0, 1.0)

def checkerboard_cell(n):
    """unit square with a 2 x 2 checkerboard (phase 2 in the lower-left and upper-right quarters), n x n cells per quarter"""
    msh = mesh.create_unit_square(comm, 2 * n, 2 * n, mesh.CellType.quadrilateral)
    return msh, indicator(msh, lambda x: (x[0] < 0.5) == (x[1] < 0.5)), (1.0, 1.0)

def disc_cell(f, h, n=1, hex_=False, void=False, order=1):
    """Gmsh mesh, periodic (matching nodes on opposite sides). Square array: n x n discs of spacing 1 (cell n x n); hexagonal array: the
    rectangle 1 x sqrt(3) with a disc at the centre and quarter discs at the corners. f = area fraction of the discs; void: no mesh in the discs."""
    import gmsh
    from dolfinx.io import gmsh as dg
    Lx, Ly = (1.0, np.sqrt(3.0)) if hex_ else (float(n), float(n))
    centres = [(0.5, np.sqrt(3) / 2), (0, 0), (1, 0), (0, np.sqrt(3)), (1, np.sqrt(3))] if hex_ else [(i + 0.5, j + 0.5) for i in range(n) for j in range(n)]
    R = np.sqrt(f * (np.sqrt(3) / 2 if hex_ else 1.0) / np.pi)               # area fraction f in every period
    gmsh.initialize(); gmsh.option.setNumber("General.Verbosity", 0); gmsh.model.add("cell")
    if comm.rank == 0:
        occ = gmsh.model.occ
        box = occ.addRectangle(0, 0, 0, Lx, Ly); discs = [(2, occ.addDisk(cx, cy, 0, R, R)) for cx, cy in centres]
        if void:
            occ.cut([(2, box)], discs)
        else:
            occ.fragment([(2, box)], discs)
        occ.synchronize()
        for _, tag in gmsh.model.getEntities(2):                     # drop the parts of the corner discs outside the box
            cx, cy, _ = occ.getCenterOfMass(2, tag)
            if not (0 < cx < Lx and 0 < cy < Ly):
                occ.remove([(2, tag)], recursive=True)
        occ.synchronize()
        surfs = [t for _, t in gmsh.model.getEntities(2)]
        pm = [0.5, 0.0, 0.0] if hex_ else [0.0, 0.0, 0.0]               # a point that is surely in the matrix
        outside = [t for t in surfs if gmsh.model.isInside(2, t, pm)]
        gmsh.model.addPhysicalGroup(2, outside, 1)
        if not void:
            gmsh.model.addPhysicalGroup(2, [t for t in surfs if t not in outside], 2)
        # periodic sides: OCC bounding boxes are enlarged by ~1e-7 (compare with 1e-6); pair the pieces of opposite sides in order
        side = lambda i, v: sorted([t for _, t in gmsh.model.getEntities(1) if abs(gmsh.model.getBoundingBox(1, t)[i] - v) < 1e-6
                                    and abs(gmsh.model.getBoundingBox(1, t)[i + 3] - v) < 1e-6], key=lambda t: gmsh.model.getBoundingBox(1, t)[1 - i])
        gmsh.model.mesh.setPeriodic(1, side(0, Lx), side(0, 0.0), [1, 0, 0, Lx, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1])
        gmsh.model.mesh.setPeriodic(1, side(1, Ly), side(1, 0.0), [1, 0, 0, 0, 0, 1, 0, Ly, 0, 0, 1, 0, 0, 0, 0, 1])
        gmsh.option.setNumber("Mesh.CharacteristicLengthMax", h); gmsh.option.setNumber("Mesh.CharacteristicLengthMin", h)
        gmsh.model.mesh.generate(2)
        if order > 1:
            gmsh.model.mesh.setOrder(order)
    md = dg.model_to_mesh(gmsh.model, comm, 0, gdim=2); gmsh.finalize()
    msh, ct = md.mesh, md.cell_tags
    ph = fem.Function(fem.functionspace(msh, ("DG", 0))); ph.x.array[ct.indices[ct.values == 2]] = 1.0; ph.x.scatter_forward()
    return msh, ph, (Lx, Ly)

# ---------------------------------------------------------------- materials and references
def lame(E, nu):
    return E * nu / ((1 + nu) * (1 - 2 * nu)), E / (2 * (1 + nu))

def C_iso(E, nu):
    """plane-strain Voigt matrix (engineering shear)"""
    lam, mu = lame(E, nu)
    return np.array([[lam + 2 * mu, lam, 0.0], [lam, lam + 2 * mu, 0.0], [0.0, 0.0, mu]])

def laminate_exact(f2, C1, C2):
    """layers normal to y, phase 2 fraction f2: e11 continuous, s22 and s12 continuous -> the exact C_hom (plane strain, Voigt)"""
    f1, out = 1 - f2, np.zeros((3, 3))
    for k in range(3):
        E = np.eye(3)[k]
        A = np.array([[f1, f2, 0, 0], [0, 0, f1, f2], [C1[1, 1], -C2[1, 1], 0, 0], [0, 0, C1[2, 2], -C2[2, 2]]])
        e1, e2, g1, g2 = np.linalg.solve(A, [E[1], E[2], -(C1[1, 0] - C2[1, 0]) * E[0], 0.0])
        out[:, k] = f1 * (C1 @ [E[0], e1, g1]) + f2 * (C2 @ [E[0], e2, g2])
    return out

def hs_bounds(f, E2, nu2, E1, nu1):
    """2D Hashin-Shtrikman bounds (plane strain, transverse) on K = (C11 + C12)/2 and G = C66, phase 2 (fraction f) stiffer than phase 1"""
    l2, g2 = lame(E2, nu2); l1, g1 = lame(E1, nu1); K2, K1 = l2 + g2, l1 + g1
    KL = K1 + f / (1 / (K2 - K1) + (1 - f) / (K1 + g1)); KU = K2 + (1 - f) / (1 / (K1 - K2) + f / (K2 + g2))
    GL = g1 + f / (1 / (g2 - g1) + (1 - f) * (K1 + 2 * g1) / (2 * g1 * (K1 + g1)))
    GU = g2 + (1 - f) / (1 / (g1 - g2) + f * (K2 + 2 * g2) / (2 * g2 * (K2 + g2)))
    return (KL, KU), (GL, GU)

def maxwell_garnett(f, k1, k2):
    """2D Maxwell-Garnett / Hashin-Shtrikman bound for discs (k2, fraction f) in a matrix (k1): exact for dilute, a bound otherwise"""
    b = (k2 - k1) / (k2 + k1)
    return k1 * (1 + f * b) / (1 - f * b)

# ---------------------------------------------------------------- periodic constraint
def interior_pin(V, L):
    """dofs at the mesh vertex nearest the cell centre that is not on the boundary (the rigid translation of a periodic cell problem)"""
    msh = V.mesh; nv = msh.topology.index_map(0).size_local
    x = msh.geometry.x[:, :2]; c = np.array(L) / 2
    ok = (x[:, 0] > TOL) & (x[:, 0] < L[0] - TOL) & (x[:, 1] > TOL) & (x[:, 1] < L[1] - TOL)
    d = np.where(ok, np.linalg.norm(x - c, axis=1), np.inf); i = int(np.argmin(d)) if len(d) else -1
    best = comm.allreduce((float(d[i]) if i >= 0 else np.inf, comm.rank), op=MPI.MINLOC)
    p = comm.bcast(x[i].copy() if comm.rank == best[1] else None, root=best[1])
    return fem.locate_dofs_geometrical(V, lambda y: np.isclose(y[0], p[0], atol=1e-12) & np.isclose(y[1], p[1], atol=1e-12))

def corner_pin(V):
    return fem.locate_dofs_geometrical(V, lambda x: np.isclose(x[0], 0, atol=TOL) & np.isclose(x[1], 0, atol=TOL))

def periodic_mpc(V, L, bcs=(), corners="once"):
    """v(x + Lx e1) = v(x), v(x + Ly e2) = v(x). corners = "once": right side, top side without its right end, then the corner (Lx, Ly)
    -> (0, 0); "twice" (the source): both sides whole, so the corner is constrained twice."""
    L1, L2 = L; on = lambda a, b: np.isclose(a, b, atol=TOL)
    mpc = dolfinx_mpc.MultiPointConstraint(V)
    if corners == "once":
        mpc.create_periodic_constraint_geometrical(V, lambda x: on(x[0], L1) & ~on(x[1], L2), lambda x: np.array([x[0] - L1, x[1], x[2]]), list(bcs))
        mpc.create_periodic_constraint_geometrical(V, lambda x: on(x[1], L2) & ~on(x[0], L1), lambda x: np.array([x[0], x[1] - L2, x[2]]), list(bcs))
        mpc.create_periodic_constraint_geometrical(V, lambda x: on(x[0], L1) & on(x[1], L2), lambda x: np.array([x[0] - L1, x[1] - L2, x[2]]), list(bcs))
    else:
        mpc.create_periodic_constraint_geometrical(V, lambda x: on(x[0], L1), lambda x: np.array([x[0] - L1, x[1], x[2]]), list(bcs))
        mpc.create_periodic_constraint_geometrical(V, lambda x: on(x[1], L2), lambda x: np.array([x[0], x[1] - L2, x[2]]), list(bcs))
    mpc.finalize()
    return mpc

# ---------------------------------------------------------------- cell problems
class _Cell:
    count = 0
    def _setup(self, msh, L, bc, pin, corners, zero):
        self.msh, self.L, self.bc = msh, L, bc
        self.vol = L[0] * L[1]; self.dx = ufl.Measure("dx", domain=msh)
        self.area = comm.allreduce(fem.assemble_scalar(fem.form(1.0 * self.dx)), op=MPI.SUM)
        fdim = msh.topology.dim - 1; msh.topology.create_connectivity(fdim, fdim + 1); V = self.V; self.mpc = None
        # the outer box only (a mesh with voids has more exterior facets: the void boundaries are traction / flux free)
        box = mesh.locate_entities_boundary(msh, fdim, lambda x: np.isclose(x[0], 0, atol=TOL) | np.isclose(x[0], L[0], atol=TOL)
                                            | np.isclose(x[1], 0, atol=TOL) | np.isclose(x[1], L[1], atol=TOL))
        self.dsb = ufl.Measure("ds", domain=msh, subdomain_data=mesh.meshtags(msh, fdim, np.sort(box), np.ones(len(box), dtype=np.int32)))(1)
        if bc == "periodic":
            self.bcs = [fem.dirichletbc(zero, interior_pin(V, L) if pin == "interior" else corner_pin(V), V)]
            self.mpc = periodic_mpc(V, L, self.bcs, corners)
        elif bc == "kinematic":
            self.bcs = [fem.dirichletbc(zero, fem.locate_dofs_topological(V, fdim, box), V)]
        else:
            self.bcs = [fem.dirichletbc(zero, corner_pin(V), V)]

    def _solve(self, L_):
        _Cell.count += 1; pre = f"cell{_Cell.count}_"
        if self.mpc is not None:
            return dolfinx_mpc.LinearProblem(self.a, L_, self.mpc, bcs=self.bcs, petsc_options=LU, petsc_options_prefix=pre).solve()
        from dolfinx.fem.petsc import LinearProblem
        return LinearProblem(self.a, L_, bcs=self.bcs, petsc_options=LU, petsc_options_prefix=pre).solve()

    def bavg(self, expr):
        """(1 / |cell|) * integral over the outer boundary of the cell"""
        return comm.allreduce(fem.assemble_scalar(fem.form(expr * self.dsb)), op=MPI.SUM) / self.vol

    def avg(self, expr):
        """(1 / |cell|) * integral over the mesh"""
        return comm.allreduce(fem.assemble_scalar(fem.form(expr * self.dx)), op=MPI.SUM) / self.vol

class ElasticCell(_Cell):
    def __init__(self, msh, ph, L, C1, C2, deg=2, bc="periodic", pin="interior", corners="once"):
        """C1 (ph = 0) and C2 (ph = 1): isotropic plane-strain Voigt matrices (C_iso)"""
        self.V = fem.functionspace(msh, ("Lagrange", deg, (2,)))
        self.lam = C1[0, 1] + (C2[0, 1] - C1[0, 1]) * ph; self.mu = C1[2, 2] + (C2[2, 2] - C1[2, 2]) * ph
        self._setup(msh, L, bc, pin, corners, np.zeros(2))
        if bc == "static":                                           # the static problem also needs the rotation fixed
            Vy, _ = self.V.sub(1).collapse()
            d = fem.locate_dofs_geometrical((self.V.sub(1), Vy), lambda x: np.isclose(x[0], L[0], atol=TOL) & np.isclose(x[1], 0, atol=TOL))
            self.bcs.append(fem.dirichletbc(fem.Function(Vy), d, self.V.sub(1)))
        self.f2 = comm.allreduce(fem.assemble_scalar(fem.form(ph * self.dx)), op=MPI.SUM) / self.vol
        u, w = ufl.TrialFunction(self.V), ufl.TestFunction(self.V)
        self.a = ufl.inner(self.sig(ufl.sym(ufl.grad(u))), ufl.sym(ufl.grad(w))) * self.dx

    def sig(self, e):
        return self.lam * ufl.tr(e) * ufl.Identity(2) + 2 * self.mu * e

    def C_hom(self):
        """3 x 3 effective matrix; columns = load cases E11, E22, 2 E12 = 1 (static BC: the inverse of the mean-strain compliance)"""
        w = ufl.TestFunction(self.V); out = np.zeros((3, 3)); self.fields = []
        for k in range(3):
            Ek = np.eye(3)[k]
            if self.bc == "static":
                S = ufl.as_matrix([[Ek[0], Ek[2]], [Ek[2], Ek[1]]]); n = ufl.FacetNormal(self.msh)
                u = self._solve(ufl.dot(ufl.dot(S, n), w) * self.dsb)
                e = ufl.sym(ufl.outer(u, n))                             # mean strain = (1/|Y|) * integral of sym(u n) over the OUTER boundary
                out[:, k] = [self.bavg(e[0, 0]), self.bavg(e[1, 1]), 2 * self.bavg(e[0, 1])]
            else:
                E = ufl.as_matrix([[Ek[0], Ek[2] / 2], [Ek[2] / 2, Ek[1]]])
                v = self._solve(-ufl.inner(self.sig(E), ufl.sym(ufl.grad(w))) * self.dx); s = self.sig(E + ufl.sym(ufl.grad(v)))
                out[:, k] = [self.avg(s[0, 0]), self.avg(s[1, 1]), self.avg(s[0, 1])]; self.fields.append(v)
        return np.linalg.inv(out) if self.bc == "static" else out

class ConductionCell(_Cell):
    def __init__(self, msh, ph, L, k1, k2, deg=2, bc="periodic", pin="interior", corners="once"):
        self.V = fem.functionspace(msh, ("Lagrange", deg))
        self.k = k1 + (k2 - k1) * ph
        self._setup(msh, L, bc, pin, corners, 0.0)
        u, w = ufl.TrialFunction(self.V), ufl.TestFunction(self.V)
        self.a = self.k * ufl.dot(ufl.grad(u), ufl.grad(w)) * self.dx

    def K_hom(self):
        """2 x 2 effective conductivity; columns = mean gradients G = e_x, e_y (static BC: uniform normal flux, inverse of the mean gradient)"""
        w = ufl.TestFunction(self.V); out = np.zeros((2, 2)); self.fields = []
        for k in range(2):
            G = ufl.as_vector(list(np.eye(2)[k]))
            if self.bc == "static":
                phi = self._solve(ufl.dot(G, ufl.FacetNormal(self.msh)) * w * self.dsb)
                n = ufl.FacetNormal(self.msh)                            # mean gradient = (1/|Y|) * integral of phi n over the OUTER boundary
                out[:, k] = [self.bavg(phi * n[0]), self.bavg(phi * n[1])]  # (with voids, the integral of grad phi over the solid is NOT |Y| <grad phi>)
            else:
                phi = self._solve(-self.k * ufl.dot(G, ufl.grad(w)) * self.dx); q = self.k * (G + ufl.grad(phi))
                out[:, k] = [self.avg(q[0]), self.avg(q[1])]
            self.fields.append(phi)
        return np.linalg.inv(out) if self.bc == "static" else out

## Q1 — Theory (15 pts)

**(a) (5)** In the shell model, which rotation component appears in none of $\boldsymbol\varepsilon$, $\boldsymbol\kappa$, $\boldsymbol\gamma$? Show it for a flat cell in the plane $z=0$. Why does the
drilling penalty $\frac{Et^3}{h^2}\,d^2$ with $d=\frac12\big((\mathbf P^T\tilde\nabla\mathbf u)_{12}-(\mathbf P^T\tilde\nabla\mathbf u)_{21}\big)+\theta_3$ not change the solution of a flat plate under a
transverse load (lab §2: $10^{-11}$), while it changes the Scordelis–Lo roof by 0.2 % (lecture)?

**(b) (5)** Show that $\langle\boldsymbol\varepsilon(\mathbf v)\rangle=\frac1{|Y|}\int_Y\boldsymbol\varepsilon(\mathbf v)\,dY=0$ for a periodic $\mathbf v$ on a rectangular cell. Then show that it also holds on a
cell with voids, if $|Y|$ is the cell area and the integral includes the void boundaries — and why that makes "divide by the meshed area" wrong.

**(c) (5)** Using the minimum potential energy, show $\mathbf E:\mathbb C^{\rm kin}\mathbf E\ge\mathbf E:\mathbb C^{\rm per}\mathbf E$. Explain (no proof) why the static BC gives the opposite inequality and why both
converge to the periodic result as the window of $n\times n$ cells grows.

> *your answer*

## Q2 — A pressurized tube: the flat-facet error (12 pts)

An open tube ($R=1$, length 2, $t=0.01$, $E=1$, $\nu=0.3$) under internal pressure $p=1$, free ends. Membrane theory: $u_r=pR^2/(Et)$, $u_z=-\nu pRz/(Et)$ — no bending.
Remove the rigid motions with point constraints that the exact solution satisfies: $u_z=0$ on the bottom ring, $u_x=0$ at $(0,\pm1,-1)$, $u_y=0$ at $(\pm1,0,-1)$.

**(a) (7)** Compute $u_r(1,0,0)/(pR^2/Et)$ for $n_\theta$ = 16, 32, 64, 128 (with $n_z=8$) and $n_\theta=32$ with $n_z$ = 4, 8, 16. Report the error and its rate in $n_\theta$ and the effect of $n_z$.

**(b) (5)** Explain the rate from the geometry: the mesh is a polygonal prism inscribed in the circle. Estimate the relative difference between the polygon and the circle
(e.g. the distance of an edge midpoint from the axis, $R\cos(\pi/n_\theta)$) and compare its size with the measured error. Which change of the model would remove it?

In [ ]:
%%fenicsx -np 2
# TODO: Q2(a) — u_r / exact for n_theta = 16, 32, 64, 128 (n_z = 8) and n_z = 4, 8, 16 (n_theta = 32)

> *your answer (Q2)*

## Q3 — Clamping an I-beam: what a marker selects (15 pts)

The I-beam of the lab (80 cells along the span, 8 across each flange, 8 over the web), clamped at both ends, 1 kN/m.

**(a) (6)** Clamp with the marker `lambda x: (x[0] < 0.3) | (x[0] > L - 0.3)` (a "box" around the ends, as in the source) and with the end facets only (`np.isclose(x[0], 0) | np.isclose(x[0], L)`).
Print the number of facets each selects and the midpoints of the facets that the box adds. Which facets are they?

**(b) (5)** Compare the midspan deflection (over the Timoshenko value) for: end facets with $\mathbf u=\boldsymbol\theta=0$; the box with $\mathbf u=\boldsymbol\theta=0$; end facets with $\mathbf u=0$ only.

**(c) (4)** Explain the size of the box effect (which deformation does clamping the flange edges near the ends prevent? estimate an effective span), and why fixing only $\mathbf u$ changes so little here.
Write a marker for the end sections that does not depend on a tolerance.

In [ ]:
%%fenicsx -np 2
# TODO: Q3 — facets selected by the two markers, the added facets' midpoints, and the three deflections

> *your answer (Q3)*

## Q4 — The Scordelis–Lo roof on a quarter with symmetry conditions (15 pts)

The roof is symmetric about $x=0$ and about $y=0$. Model the quarter $0<x<25$, $0<\text{angle}<40°$ (`scordelis_lo_mesh(N, quarter=True)`).

**(a) (5)** Derive the symmetry conditions on a plane with unit normal $\mathbf n$ for the displacement and for the rotation **vector** $\boldsymbol\theta$, and write them for the planes $x=0$ and $y=0$
in Cartesian components. (Hint: a rotation vector is a pseudo-vector.)

**(b) (6)** Solve the quarter with $N$ = 4, 8, 16, 32 and compare with the full roof at the same cell size ($N$ = 8, 16, 32, 64): the free-edge displacement and the run time.

**(c) (4)** What would you see if you forgot the rotation conditions on the symmetry planes? Try it for $N=16$.

In [ ]:
%%fenicsx -np 2
# TODO: Q4(b) — quarter (with symmetry conditions) vs full roof at the same cell size; Q4(c) — the quarter without the rotation conditions

> *your answer (Q4)*

## Q5 — The checkerboard: a corner singularity and a graded mesh (15 pts)

A 2×2 checkerboard of conductivities $k_1$, $k_2$ has $k_{\rm eff}=\sqrt{k_1k_2}$ exactly (Dykhne). At a corner where the four squares meet, the field behaves like $r^\lambda$ with
$\lambda=\frac4\pi\arctan\sqrt{k_1/k_2}$ ($k_1<k_2$), and the error of the energy (hence of $k_{\rm eff}$) decreases like $h^{2\lambda}$ on uniform meshes.

**(a) (6)** Measure the rate of $|k/\sqrt{k_1k_2}-1|$ on uniform meshes ($n$ = 8, 16, 32, 64 cells per quarter) for $k_2/k_1$ = 10 and 100 and compare with $2\lambda$.

**(b) (6)** Grade the mesh towards the lines $x,y\in\{0,\tfrac12,1\}$ (all corners lie on them): in every half $[0,\tfrac12]$ map $s\mapsto g(s)$ with $g(s)=\tfrac12(2s)^p$ for $s<\tfrac12$ and symmetric above
(acting on the fraction inside each half). Use $p=2$ and repeat (a) for $k_2/k_1=10$. What rate do you get, and why does grading help?

**(c) (3)** For the checkerboard, what does Keller's theorem (lab §7) reduce to, and what does that mean for using it as a check here?

In [ ]:
%%fenicsx -np 2
# TODO: Q5 — uniform and graded checkerboard meshes, errors and rates

> *your answer (Q5)*

## Q6 — A porous medium: cell BCs and window size (13 pts)

Insulating circular pores (porosity $\phi=0.3$, square array) in a solid of conductivity 1 — e.g. the solid phase of a porous electrode. `disc_cell(0.3, h, n=n, void=True, order=2)` meshes a window of
$n\times n$ pores (only the solid).

**(a) (7)** Compute $k_{\rm eff}$ with the kinematic, periodic and static BCs for $n$ = 1, 2, 4 ($h=0.05$). Check the order of the three and the Hashin–Shtrikman upper bound $(1-\phi)/(1+\phi)$.

**(b) (6)** How large a window would the kinematic and the static BC need for 1 % accuracy? What do you conclude for a micro-CT image of a real electrode (no periodicity)?

In [ ]:
%%fenicsx -np 2
# TODO: Q6 — kinematic / periodic / static for windows n = 1, 2, 4

> *your answer (Q6)*

## Q7 — Debugging an LLM-generated homogenization script (15 pts)

A student asked an LLM for "the effective conductivity of a porous electrode with circular pores (porosity 0.4) by periodic homogenization with dolfinx_mpc, verified against theory".
The answer is below, as received. It prints a result and declares it verified.

**(a) (3)** Run it (with `-np 2` as given). What does it print, and is the claimed verification meaningful?

**(b) (8)** Find the defects (there are at least four, some silent). For each: what is wrong, what it does to the result (measure it by fixing the others), and which test reveals it.

**(c) (4)** Write the corrected script (or use `homog.py`) and verify the result with tests that do not need the answer: the Hashin–Shtrikman bound, symmetry of the tensor, mesh refinement, `-np 1` vs `-np 2`.

In [ ]:
%%fenicsx -np 2
# LLM-generated (as received): effective conductivity of a porous electrode (solid matrix, circular pores, square array)
import numpy as np, ufl, gmsh
from mpi4py import MPI
from dolfinx import fem, mesh
from dolfinx.io import gmsh as dg
import dolfinx_mpc
comm = MPI.COMM_WORLD
phi_p, ks = 0.4, 1.0                                                     # porosity, solid conductivity
R = np.sqrt(phi_p / np.pi)
gmsh.initialize(); gmsh.option.setNumber("General.Verbosity", 0)
if comm.rank == 0:
    occ = gmsh.model.occ
    box = occ.addRectangle(0, 0, 0, 1, 1); hole = occ.addDisk(0.5, 0.5, 0, R, R)
    occ.cut([(2, box)], [(2, hole)]); occ.synchronize()
    gmsh.model.addPhysicalGroup(2, [s[1] for s in gmsh.model.getEntities(2)], 1)
    gmsh.option.setNumber("Mesh.CharacteristicLengthMax", 0.04)
    gmsh.model.mesh.generate(2)
msh = dg.model_to_mesh(gmsh.model, comm, 0, gdim=2).mesh
gmsh.finalize()
V = fem.functionspace(msh, ("Lagrange", 2))
corner = fem.locate_dofs_geometrical(V, lambda x: np.isclose(x[0], 0) & np.isclose(x[1], 0))
bc = fem.dirichletbc(0.0, corner, V)                                     # fix the constant
mpc = dolfinx_mpc.MultiPointConstraint(V)
mpc.create_periodic_constraint_geometrical(V, lambda x: np.isclose(x[0], 1), lambda x: np.array([x[0] - 1, x[1], x[2]]), [bc])
mpc.create_periodic_constraint_geometrical(V, lambda x: np.isclose(x[1], 1), lambda x: np.array([x[0], x[1] - 1, x[2]]), [bc])
mpc.finalize()
u, w = ufl.TrialFunction(V), ufl.TestFunction(V)
a = ks * ufl.dot(ufl.grad(u), ufl.grad(w)) * ufl.dx
L = -ks * ufl.dot(ufl.as_vector([1.0, 0.0]), ufl.grad(w)) * ufl.dx       # macroscopic gradient e_x
phi = dolfinx_mpc.LinearProblem(a, L, mpc, bcs=[bc], petsc_options={"ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"},
                                petsc_options_prefix="llm_").solve()
area = comm.allreduce(fem.assemble_scalar(fem.form(1.0 * ufl.dx(domain=msh))), op=MPI.SUM)
k_eff = comm.allreduce(fem.assemble_scalar(fem.form(ks * (1.0 + phi.dx(0)) * ufl.dx)), op=MPI.SUM) / area
bruggeman = ks * (1 - phi_p)**1.5                                        # exact effective-medium result
print(f"k_eff / k_s = {k_eff:.5f}, Bruggeman {bruggeman:.5f}, error {abs(k_eff / bruggeman - 1) * 100:.1f} %  -> verified")

In [ ]:
%%fenicsx -np 2
# TODO: Q7(c) — the corrected script and its verification

> *your list of defects and tests (Q7b), and your verification (Q7c)*

## 🤖 AI-usage log

*Required in every assignment and project. Using an LLM is encouraged and never penalized — an empty or vague log is.*

| # | Tool / model | What I asked (key prompt, shortened) | What it got wrong or missed | How I detected it | How I fixed it |
|---|---|---|---|---|---|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |

**Code provenance** — which cells/functions were mostly generated, which did you write or substantially change?

> …

**Verification** — how do you know the final result is correct (MMS rates, benchmark value, conservation check, parallel = serial)?

> …

**Reflection (2–3 sentences)** — what would you do differently when prompting next time?

> …